# 01 — PMC Ingestion (Clean)

This notebook builds the **Traceable-RAG** corpus by downloading **PubMed Central Open Access (PMC OA)** full-text article packages and extracting their **JATS XML (`.nxml`)** files.

**Output**
- `Traceable_RAG/data/raw/nxml/` — extracted `PMCxxxxxxx.nxml` files (target: 500)
- `Traceable_RAG/data/raw/pmc_packages/` — downloaded `.tar.gz` packages

**Notes**
- We only download articles that are available in the **PMC Open Access subset** (via `oa.fcgi`).
- The pipeline is **restart-safe**: if an `.nxml` already exists, it will be skipped.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install dependencies (run once per Colab runtime)
!pip -q install biopython lxml pandas tqdm requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 33.5 MB/s eta 0:00:00


In [3]:
import re
import time
import tarfile
import random
import requests
import xml.etree.ElementTree as ET
from pathlib import Path
from urllib.error import HTTPError
from Bio import Entrez

## Configuration

1) Set your real email for NCBI Entrez usage (required).  
2) Adjust `TARGET`, `RETMAX`, and `QUERY` if needed.

**Tip:** Start with `TARGET = 50` for a quick test, then increase to 500.


In [4]:
# === REQUIRED ===
Entrez.email = "ivan.rmitkov@gmail.com"   # TODO: replace with your real email
Entrez.tool = "traceable-rag"

# Drive project root
PROJECT_ROOT = "/content/drive/MyDrive/Traceable_RAG"

# Target corpus size
TARGET = 500

# How many candidate PMC records to retrieve (OA filter will reduce this a lot)
RETMAX = 20000

# Query tuned for INSTI / integrase inhibitor resistance (Title/Abstract focus)
QUERY = (
    '(HIV[Title/Abstract] AND '
    '(integrase[Title/Abstract] OR INSTI[Title/Abstract] OR "integrase strand transfer"[Title/Abstract]) AND '
    '(resist*[Title/Abstract] OR mutation*[Title/Abstract]))'
)

# Throttling (stability > speed)
SLEEP_PER_ITER = 0.6

In [5]:
# Create output folders
raw_pkg = Path(PROJECT_ROOT) / "data" / "raw" / "pmc_packages"
raw_nxml = Path(PROJECT_ROOT) / "data" / "raw" / "nxml"
raw_pkg.mkdir(parents=True, exist_ok=True)
raw_nxml.mkdir(parents=True, exist_ok=True)

print("Packages:", raw_pkg)
print("NXML:", raw_nxml)
print("Already have NXML:", len(list(raw_nxml.glob("*.nxml"))))

Packages: /content/drive/MyDrive/Traceable_RAG/data/raw/pmc_packages
NXML: /content/drive/MyDrive/Traceable_RAG/data/raw/nxml
Already have NXML: 500


## Helper functions

- `uid_to_pmcid`: Converts Entrez PMC UID → PMCID (e.g., `PMC1234567`) with retry/backoff.
- `pmcid_to_tgz_https`: Uses PMC OA API to get a `.tgz` link (only for OA articles); converts ftp→https.
- `download_and_extract_nxml`: Downloads the package and extracts the **largest** `.nxml` inside.


In [6]:
PMCID_RE = re.compile(r"(PMC\d+)")

def uid_to_pmcid(uid: str, retries: int = 6, base_sleep: float = 0.6) -> str | None:
    """Convert PMC UID to PMCID via efetch (MEDLINE) with retries."""
    for attempt in range(retries):
        try:
            h = Entrez.efetch(db="pmc", id=uid, rettype="medline", retmode="text")
            txt = h.read()
            h.close()
            m = PMCID_RE.search(txt)
            return m.group(1) if m else None
        except HTTPError as e:
            wait = base_sleep * (2 ** attempt) + random.uniform(0, 0.4)
            print(f"[uid_to_pmcid] HTTPError {e.code} for UID={uid}. retry in {wait:.2f}s")
            time.sleep(wait)
        except Exception as e:
            wait = base_sleep * (2 ** attempt) + random.uniform(0, 0.4)
            print(f"[uid_to_pmcid] Error {e} for UID={uid}. retry in {wait:.2f}s")
            time.sleep(wait)
    return None


def pmcid_to_tgz_https(pmcid: str, timeout: int = 30) -> str | None:
    """Return https tgz link for OA articles; None if not OA or not available."""
    oa_url = f"https://www.ncbi.nlm.nih.gov/pmc/utils/oa/oa.fcgi?id={pmcid}"
    resp = requests.get(oa_url, timeout=timeout)
    if resp.status_code != 200 or not resp.text:
        return None
    try:
        root = ET.fromstring(resp.text)
    except ET.ParseError:
        return None

    for link in root.findall(".//link"):
        if link.attrib.get("format") == "tgz":
            href = link.attrib.get("href")
            if not href:
                return None
            return href.replace("ftp://ftp.ncbi.nlm.nih.gov/", "https://ftp.ncbi.nlm.nih.gov/")
    return None


def download_and_extract_nxml(pmcid: str, tgz_link: str, timeout: int = 240) -> Path:
    """Download tgz and extract the largest .nxml to raw_nxml/PMCID.nxml"""
    tgz_path = raw_pkg / f"{pmcid}.tar.gz"
    nxml_path = raw_nxml / f"{pmcid}.nxml"

    r = requests.get(tgz_link, timeout=timeout)
    if r.status_code != 200 or not r.content:
        raise RuntimeError(f"download failed status={r.status_code}")

    # gzip magic bytes check
    if not (len(r.content) >= 2 and r.content[0] == 0x1f and r.content[1] == 0x8b):
        head = r.text[:200] if hasattr(r, "text") else str(r.content[:200])
        raise RuntimeError(f"not gzip (head={head!r})")

    tgz_path.write_bytes(r.content)

    with tarfile.open(tgz_path, "r:gz") as tf:
        nxml_members = [m for m in tf.getmembers() if m.name.endswith(".nxml")]
        if not nxml_members:
            raise RuntimeError("no .nxml inside tgz")
        nxml_members.sort(key=lambda x: x.size, reverse=True)
        f = tf.extractfile(nxml_members[0])
        if f is None:
            raise RuntimeError("cannot extract .nxml file")
        nxml_path.write_bytes(f.read())

    return nxml_path

## Run ingestion (OA only)

This cell:
1) Searches PMC for candidates using `QUERY`
2) Iterates from the beginning **every run**
3) Skips any PMCID that already has an `.nxml` file
4) Downloads OA packages until `TARGET` is reached

If you already have 500 NXML files, it will stop immediately.


In [7]:
# Search candidates
handle = Entrez.esearch(db="pmc", term=QUERY, retmax=RETMAX)
record = Entrez.read(handle)
handle.close()

uids = record.get("IdList", [])
print("Candidates (UIDs):", len(uids))

downloaded = len(list(raw_nxml.glob("*.nxml")))
print("Already have NXML:", downloaded)

# Main loop (always starts from the beginning of uids)
for i, uid in enumerate(uids, start=1):
    if downloaded >= TARGET:
        break

    # periodic pause to be friendly to NCBI
    if i % 200 == 0:
        print(f"--- checkpoint: processed {i}/{len(uids)}, downloaded={downloaded} ---")
        time.sleep(2.0)

    pmcid = uid_to_pmcid(uid)
    if not pmcid:
        continue

    nxml_file = raw_nxml / f"{pmcid}.nxml"
    if nxml_file.exists():
        continue  # already downloaded/extracted

    tgz_link = pmcid_to_tgz_https(pmcid)
    if not tgz_link:
        continue  # not OA (or no package)

    try:
        print("Downloading:", pmcid)
        nxml_path = download_and_extract_nxml(pmcid, tgz_link)
        downloaded += 1
        print("Saved NXML:", downloaded, "|", nxml_path.name)
    except Exception as e:
        print("Error:", pmcid, "|", e)

    time.sleep(SLEEP_PER_ITER)

print("Finished. Total NXML:", downloaded)

Candidates (UIDs): 1233
Already have NXML: 500
Finished. Total NXML: 500


In [8]:
# Final verification
import os
print("Total NXML:", len(os.listdir(raw_nxml)))

Total NXML: 500


In [10]:
%cd /content/drive/MyDrive/Traceable_RAG/traceable-rag

/content/drive/MyDrive/Traceable_RAG/traceable-rag


In [11]:
!git status

On branch main

No commits yet

nothing to commit (create/copy files and use "git add" to track)


In [13]:
!ls /content/drive/MyDrive/Traceable_RAG

data  indexes  notebooks  PMC12956220  PMC12956220.tar.gz  traceable-rag


In [15]:
import shutil

shutil.move(
"/content/drive/MyDrive/Traceable_RAG/notebooks/01_pmc_ingestion_clean.ipynb",
"/content/drive/MyDrive/Traceable_RAG/traceable-rag/01_pmc_ingestion_clean.ipynb"
)

'/content/drive/MyDrive/Traceable_RAG/traceable-rag/01_pmc_ingestion_clean.ipynb'